# NB6 · Modelo final para Power BI y diagnóstico ejecutivo

## Objetivo

Preparar las tablas definitivas del producto:

- modelo estrella sencillo para Power BI;
- histórico y forecast en una misma estructura;
- dimensiones de fechas, cuentas, ratios y escenarios;
- tabla de diagnóstico ejecutivo;
- alertas y principales causas de variación.

> Este notebook no recalcula estados financieros. Organiza las salidas anteriores para consumirlas directamente en Power BI.

In [1]:
# ==========================================================
# 1. LIBRERÍAS Y RUTAS
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


pd.set_option("display.max_columns", 120)
pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# ==========================================================
# RUTAS DEL PROYECTO
# ==========================================================

CWD = Path.cwd()

if (CWD / "data").exists():
    BASE_DIR = CWD
elif (CWD.parent / "data").exists():
    BASE_DIR = CWD.parent
else:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto."
    )


PROCESSED_DIR = BASE_DIR / "data" / "processed"
POWERBI_DIR = BASE_DIR / "data" / "powerbi"

POWERBI_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RUTAS = {
    "maestro_cuentas": (
        PROCESSED_DIR
        / "02"
        / "02_maestro_financiero.csv"
    ),

    "contabilidad": (
        PROCESSED_DIR
        / "02"
        / "02_contabilidad_financiera.csv"
    ),

    "pyg_centro": (
        PROCESSED_DIR
        / "03"
        / "03_pyg_centro_coste.csv"
    ),

    "metricas_historicas": (
        PROCESSED_DIR
        / "04"
        / "04_metricas_powerbi.csv"
    ),

    "ratios_historicos": (
        PROCESSED_DIR
        / "04"
        / "04_ratios_powerbi.csv"
    ),

    "catalogo_ratios": (
        PROCESSED_DIR
        / "04"
        / "04_catalogo_ratios.csv"
    ),

    "dupont": (
        PROCESSED_DIR
        / "04"
        / "04_dupont.csv"
    ),

    "forecast": (
        PROCESSED_DIR
        / "05"
        / "05_forecast_powerbi.csv"
    ),

    "ratios_forecast": (
        PROCESSED_DIR
        / "05"
        / "05_ratios_forecast_powerbi.csv"
    ),

    "resumen_escenarios": (
        PROCESSED_DIR
        / "05"
        / "05_resumen_escenarios.csv"
    ),
}


for nombre, ruta in RUTAS.items():

    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontró '{ruta.name}'. "
            "Ejecuta primero los notebooks anteriores."
        )


print(f"Raíz del proyecto: {BASE_DIR}")
print(f"Salida Power BI:    {POWERBI_DIR}")

print("\nArchivos encontrados:")

for nombre, ruta in RUTAS.items():
    print(f" - {nombre}: {ruta.name}")

Raíz del proyecto: C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios
Salida Power BI:    C:\Users\moned\Downloads\Data_Science_Evolve\analisis_ratios\data\powerbi

Archivos encontrados:
 - maestro_cuentas: 02_maestro_financiero.csv
 - contabilidad: 02_contabilidad_financiera.csv
 - pyg_centro: 03_pyg_centro_coste.csv
 - metricas_historicas: 04_metricas_powerbi.csv
 - ratios_historicos: 04_ratios_powerbi.csv
 - catalogo_ratios: 04_catalogo_ratios.csv
 - dupont: 04_dupont.csv
 - forecast: 05_forecast_powerbi.csv
 - ratios_forecast: 05_ratios_forecast_powerbi.csv
 - resumen_escenarios: 05_resumen_escenarios.csv


In [2]:
# ==========================================================
# 2. CARGA DE DATOS
# ==========================================================

maestro_cuentas = pd.read_csv(
    RUTAS["maestro_cuentas"],
    dtype={
        "cuenta_n1": "string",
        "cuenta_n2": "string",
        "cuenta_n5": "string",
    }
)


contabilidad = pd.read_csv(
    RUTAS["contabilidad"],
    dtype={
        "cuenta_n5": "string",
        "centro_coste": "string",
    },
    parse_dates=["periodo"]
)


pyg_centro = pd.read_csv(
    RUTAS["pyg_centro"],
    dtype={
        "centro_coste": "string",
    },
    parse_dates=["periodo"]
)


metricas_historicas = pd.read_csv(
    RUTAS["metricas_historicas"],
    parse_dates=["periodo"]
)


ratios_historicos = pd.read_csv(
    RUTAS["ratios_historicos"],
    parse_dates=["periodo"]
)


catalogo_ratios = pd.read_csv(
    RUTAS["catalogo_ratios"]
)


dupont = pd.read_csv(
    RUTAS["dupont"],
    parse_dates=["periodo"]
)


forecast = pd.read_csv(
    RUTAS["forecast"],
    parse_dates=["periodo"]
)


ratios_forecast = pd.read_csv(
    RUTAS["ratios_forecast"],
    parse_dates=["periodo"]
)


resumen_escenarios = pd.read_csv(
    RUTAS["resumen_escenarios"]
)


print("Datos cargados correctamente.")

print(
    f"Histórico financiero: "
    f"{metricas_historicas['periodo'].min():%Y-%m} "
    f"→ "
    f"{metricas_historicas['periodo'].max():%Y-%m}"
)

print(
    f"Forecast: "
    f"{forecast['periodo'].min():%Y-%m} "
    f"→ "
    f"{forecast['periodo'].max():%Y-%m}"
)

Datos cargados correctamente.
Histórico financiero: 2021-01 → 2023-12
Forecast: 2024-01 → 2024-12


In [3]:
# ==========================================================
# 3. DIMENSIÓN CALENDARIO
# ==========================================================

ultima_fecha_real = (
    metricas_historicas[
        "periodo"
    ].max()
)


fecha_min = min(
    contabilidad["periodo"].min(),
    metricas_historicas["periodo"].min()
)


fecha_max = max(
    forecast["periodo"].max(),
    ratios_forecast["periodo"].max()
)


dim_calendario = pd.DataFrame({
    "fecha": pd.date_range(
        start=fecha_min,
        end=fecha_max,
        freq="MS"
    )
})


dim_calendario["año"] = (
    dim_calendario[
        "fecha"
    ].dt.year
)


dim_calendario[
    "mes_numero"
] = (
    dim_calendario[
        "fecha"
    ].dt.month
)


MESES = {
    1: "Enero",
    2: "Febrero",
    3: "Marzo",
    4: "Abril",
    5: "Mayo",
    6: "Junio",
    7: "Julio",
    8: "Agosto",
    9: "Septiembre",
    10: "Octubre",
    11: "Noviembre",
    12: "Diciembre",
}


dim_calendario[
    "mes_nombre"
] = (
    dim_calendario[
        "mes_numero"
    ]
    .map(MESES)
)


dim_calendario[
    "año_mes"
] = (
    dim_calendario[
        "fecha"
    ]
    .dt.strftime("%Y-%m")
)


dim_calendario[
    "trimestre"
] = (
    "T"
    + dim_calendario[
        "fecha"
    ]
    .dt.quarter
    .astype(str)
)


dim_calendario[
    "orden_año_mes"
] = (
    dim_calendario["año"]
    * 100
    + dim_calendario[
        "mes_numero"
    ]
)


dim_calendario[
    "tipo_periodo"
] = np.where(
    dim_calendario[
        "fecha"
    ] <= ultima_fecha_real,
    "Histórico",
    "Forecast"
)


display(
    dim_calendario.tail()
)

,fecha,año,mes_numero,mes_nombre,año_mes,trimestre,orden_año_mes,tipo_periodo
43,2024-08-01,2024,8,Agosto,2024-08,T3,202408,Forecast
44,2024-09-01,2024,9,Septiembre,2024-09,T3,202409,Forecast
45,2024-10-01,2024,10,Octubre,2024-10,T4,202410,Forecast
46,2024-11-01,2024,11,Noviembre,2024-11,T4,202411,Forecast
47,2024-12-01,2024,12,Diciembre,2024-12,T4,202412,Forecast


In [4]:
# ==========================================================
# 4. DIMENSIONES DEL MODELO
# ==========================================================


# ==========================================================
# CUENTAS
# ==========================================================

dim_cuentas = (
    maestro_cuentas
    .drop_duplicates(
        "cuenta_n5"
    )
    .copy()
)

# ==========================================================
# CENTROS DE COSTE
# ==========================================================

dim_centros_coste = (
    contabilidad[[
        "centro_coste"
    ]]
    .copy()
)

dim_centros_coste[
    "centro_coste"
] = (
    dim_centros_coste[
        "centro_coste"
    ]
    .astype("string")
    .fillna("SIN CENTRO")
    .str.strip()
    .replace(
        "",
        "SIN CENTRO"
    )
)

dim_centros_coste = (
    dim_centros_coste
    .drop_duplicates()
    .sort_values(
        "centro_coste"
    )
    .reset_index(drop=True)
)

dim_centros_coste[
    "centro_etiqueta"
] = [
    "SIN CENTRO"
    if centro == "SIN CENTRO"
    else f"CC{i + 1:02d}"
    for i, centro in enumerate(
        dim_centros_coste[
            "centro_coste"
        ]
    )
]


# ==========================================================
# RATIOS
# ==========================================================

dim_ratios = (
    catalogo_ratios
    .drop_duplicates(
        "ratio"
    )
    .copy()
)


# ==========================================================
# ESCENARIOS
# ==========================================================

dim_escenarios = pd.DataFrame({
    "escenario": [
        "Real",
        "Pesimista",
        "Base",
        "Optimista",
    ],

    "orden": [
        0,
        1,
        2,
        3,
    ],

    "tipo": [
        "Histórico",
        "Forecast",
        "Forecast",
        "Forecast",
    ],
})


print(
    f"Cuentas:          {len(dim_cuentas):,}"
)

print(
    f"Centros de coste: {len(dim_centros_coste):,}"
)

print(
    f"Ratios:           {len(dim_ratios):,}"
)

display(dim_escenarios)

Cuentas:          1,565
Centros de coste: 30
Ratios:           24


,escenario,orden,tipo
0,Real,0,Histórico
1,Pesimista,1,Forecast
2,Base,2,Forecast
3,Optimista,3,Forecast


In [5]:
# ==========================================================
# 5. FACT FINANZAS HISTÓRICA
# ==========================================================

fact_finanzas_historica = (
    metricas_historicas
    .rename(
        columns={
            "periodo":
                "fecha",

            "variable":
                "metrica",
        }
    )
    .copy()
)


fact_finanzas_historica[
    "escenario"
] = "Real"


fact_finanzas_historica[
    "tipo_periodo"
] = "Histórico"


# ==========================================================
# ORDEN DE COLUMNAS
# ==========================================================

fact_finanzas_historica = (
    fact_finanzas_historica[[
        "fecha",
        "metrica",
        "horizonte",
        "valor",
        "valor_ly",
        "variacion_abs",
        "variacion_pct",
        "escenario",
        "tipo_periodo",
        "año",
        "mes_numero",
        "año_mes",
    ]]
)


display(
    fact_finanzas_historica.head()
)

,fecha,metrica,horizonte,valor,valor_ly,variacion_abs,variacion_pct,escenario,tipo_periodo,año,mes_numero,año_mes
0,2021-01-01,activo_corriente,CIERRE,"2,987,136.88",NaN,NaN,NaN,Real,Histórico,2021,1,2021-01
1,2021-01-01,activo_no_corriente,CIERRE,"21,322,881.77",NaN,NaN,NaN,Real,Histórico,2021,1,2021-01
2,2021-01-01,activo_total,CIERRE,"24,310,018.65",NaN,NaN,NaN,Real,Histórico,2021,1,2021-01
3,2021-01-01,clientes,CIERRE,"83,348.18",NaN,NaN,NaN,Real,Histórico,2021,1,2021-01
4,2021-01-01,deuda_financiera_bruta,CIERRE,"23,278,079.20",NaN,NaN,NaN,Real,Histórico,2021,1,2021-01


In [6]:
# ==========================================================
# 6. FACT FINANZAS HISTÓRICO + FORECAST
# ==========================================================

# ==========================================================
# HORIZONTE DEL FORECAST
# ==========================================================

metricas_flujo = {
    "ingresos_explotacion",
    "margen_bruto",
    "ebitda",
    "ebit",
    "resultado_neto",
}


fact_forecast = (
    forecast
    .rename(
        columns={
            "periodo":
                "fecha",

            "metrica":
                "metrica",
        }
    )
    .copy()
)


fact_forecast[
    "horizonte"
] = np.where(
    fact_forecast[
        "metrica"
    ].isin(
        metricas_flujo
    ),
    "MTD",
    "CIERRE"
)


fact_forecast[
    "valor_ly"
] = np.nan

fact_forecast[
    "variacion_abs"
] = np.nan

fact_forecast[
    "variacion_pct"
] = np.nan

fact_forecast[
    "tipo_periodo"
] = "Forecast"


fact_forecast = (
    fact_forecast[[
        "fecha",
        "metrica",
        "horizonte",
        "valor",
        "valor_ly",
        "variacion_abs",
        "variacion_pct",
        "escenario",
        "tipo_periodo",
        "año",
        "mes_numero",
        "año_mes",
    ]]
)


# ==========================================================
# UNIÓN
# ==========================================================

fact_finanzas = pd.concat(
    [
        fact_finanzas_historica,
        fact_forecast,
    ],
    ignore_index=True
)


fact_finanzas = (
    fact_finanzas
    .sort_values(
        [
            "fecha",
            "metrica",
            "horizonte",
            "escenario",
        ]
    )
    .reset_index(drop=True)
)


display(
    fact_finanzas.tail()
)

,fecha,metrica,horizonte,valor,valor_ly,variacion_abs,variacion_pct,escenario,tipo_periodo,año,mes_numero,año_mes
1471,2024-12-01,resultado_neto,MTD,"249,439.06",NaN,NaN,NaN,Optimista,Forecast,2024,12,2024-12
1472,2024-12-01,resultado_neto,MTD,"69,978.79",NaN,NaN,NaN,Pesimista,Forecast,2024,12,2024-12
1473,2024-12-01,tesoreria,CIERRE,"6,889,626.64",NaN,NaN,NaN,Base,Forecast,2024,12,2024-12
1474,2024-12-01,tesoreria,CIERRE,"7,578,589.31",NaN,NaN,NaN,Optimista,Forecast,2024,12,2024-12
1475,2024-12-01,tesoreria,CIERRE,"6,200,663.98",NaN,NaN,NaN,Pesimista,Forecast,2024,12,2024-12


In [7]:
# ==========================================================
# 7. FACT RATIOS HISTÓRICO + FORECAST
# ==========================================================

# ==========================================================
# HISTÓRICO
# ==========================================================

fact_ratios_historicos = (
    ratios_historicos
    .rename(
        columns={
            "periodo":
                "fecha",

            "valor_año_anterior":
                "valor_ly",

            "variacion_anual":
                "variacion_abs",

            "variacion_anual_pct":
                "variacion_pct",
        }
    )
    .copy()
)


fact_ratios_historicos[
    "escenario"
] = "Real"


fact_ratios_historicos[
    "tipo_periodo"
] = "Histórico"


columnas_fact_ratios = [
    "fecha",
    "ratio",
    "horizonte",
    "valor",
    "valor_ly",
    "variacion_abs",
    "variacion_pct",
    "estado",
    "escenario",
    "tipo_periodo",
    "año",
    "mes_numero",
    "año_mes",
]


fact_ratios_historicos = (
    fact_ratios_historicos[
        columnas_fact_ratios
    ]
)


# ==========================================================
# FORECAST
# ==========================================================

fact_ratios_forecast = (
    ratios_forecast
    .rename(
        columns={
            "periodo":
                "fecha"
        }
    )
    .copy()
)


ratios_mtd = {
    "margen_bruto",
    "margen_ebitda",
    "margen_ebit",
    "margen_neto",
}


fact_ratios_forecast[
    "horizonte"
] = np.where(
    fact_ratios_forecast[
        "ratio"
    ].isin(
        ratios_mtd
    ),
    "MTD",
    "CIERRE"
)


fact_ratios_forecast[
    "valor_ly"
] = np.nan

fact_ratios_forecast[
    "variacion_abs"
] = np.nan

fact_ratios_forecast[
    "variacion_pct"
] = np.nan

fact_ratios_forecast[
    "estado"
] = "FORECAST"

fact_ratios_forecast[
    "tipo_periodo"
] = "Forecast"


fact_ratios_forecast = (
    fact_ratios_forecast[
        columnas_fact_ratios
    ]
)


# ==========================================================
# UNIÓN
# ==========================================================

fact_ratios = pd.concat(
    [
        fact_ratios_historicos,
        fact_ratios_forecast,
    ],
    ignore_index=True
)


fact_ratios = (
    fact_ratios
    .sort_values(
        [
            "fecha",
            "ratio",
            "horizonte",
            "escenario",
        ]
    )
    .reset_index(drop=True)
)


display(
    fact_ratios.tail()
)

,fecha,ratio,horizonte,valor,valor_ly,variacion_abs,variacion_pct,estado,escenario,tipo_periodo,año,mes_numero,año_mes
1507,2024-12-01,prueba_acida,CIERRE,2.59,NaN,NaN,NaN,FORECAST,Optimista,Forecast,2024,12,2024-12
1508,2024-12-01,prueba_acida,CIERRE,2.11,NaN,NaN,NaN,FORECAST,Pesimista,Forecast,2024,12,2024-12
1509,2024-12-01,tesoreria_sobre_activo,CIERRE,0.27,NaN,NaN,NaN,FORECAST,Base,Forecast,2024,12,2024-12
1510,2024-12-01,tesoreria_sobre_activo,CIERRE,0.29,NaN,NaN,NaN,FORECAST,Optimista,Forecast,2024,12,2024-12
1511,2024-12-01,tesoreria_sobre_activo,CIERRE,0.25,NaN,NaN,NaN,FORECAST,Pesimista,Forecast,2024,12,2024-12


In [8]:
# ==========================================================
# 8. FACT CENTROS DE COSTE
# ==========================================================

fact_centros_coste = (
    pyg_centro
    .rename(
        columns={
            "periodo":
                "fecha",

            "importe_mes":
                "valor_mes",

            "importe_ytd":
                "valor_ytd",
        }
    )
    .copy()
)


fact_centros_coste[
    "centro_coste"
] = (
    fact_centros_coste[
        "centro_coste"
    ]
    .astype("string")
    .fillna("SIN CENTRO")
    .str.strip()
    .replace(
        "",
        "SIN CENTRO"
    )
)


fact_centros_coste[
    "escenario"
] = "Real"


fact_centros_coste[
    "año"
] = (
    fact_centros_coste[
        "fecha"
    ]
    .dt.year
)


fact_centros_coste[
    "mes_numero"
] = (
    fact_centros_coste[
        "fecha"
    ]
    .dt.month
)


fact_centros_coste[
    "año_mes"
] = (
    fact_centros_coste[
        "fecha"
    ]
    .dt.strftime("%Y-%m")
)


display(
    fact_centros_coste.head()
)

,fecha,centro_coste,categoria_financiera,subcategoria_financiera,driver_negocio,valor_mes,año,mes_numero,año_mes,valor_ytd,escenario
0,2021-01-01,G001,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-7,387.34",2021,1,2021-01,"-7,387.34",Real
1,2021-02-01,G001,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-7,278.81",2021,2,2021-02,"-14,666.15",Real
2,2021-03-01,G001,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-7,041.78",2021,3,2021-03,"-21,707.93",Real
3,2021-04-01,G001,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-7,074.80",2021,4,2021-04,"-28,782.73",Real
4,2021-05-01,G001,Amortizaciones,Dotaciones para amortizaciones,Amortización,"-6,813.84",2021,5,2021-05,"-35,596.57",Real


In [9]:
# ==========================================================
# 9. FACT CONTABILIDAD Y FACT DUPONT
# ==========================================================

# ==========================================================
# CONTABILIDAD
# ==========================================================

fact_contabilidad = (
    contabilidad
    .rename(
        columns={
            "periodo":
                "fecha"
        }
    )
    .copy()
)


fact_contabilidad[
    "centro_coste"
] = (
    fact_contabilidad[
        "centro_coste"
    ]
    .astype("string")
    .fillna("SIN CENTRO")
    .str.strip()
    .replace(
        "",
        "SIN CENTRO"
    )
)


fact_contabilidad[
    "escenario"
] = "Real"


fact_contabilidad[
    "tipo_periodo"
] = "Histórico"


# ==========================================================
# DUPONT
# ==========================================================

fact_dupont = (
    dupont
    .rename(
        columns={
            "periodo":
                "fecha"
        }
    )
    .copy()
)


fact_dupont[
    "año"
] = (
    fact_dupont[
        "fecha"
    ]
    .dt.year
)


fact_dupont[
    "mes_numero"
] = (
    fact_dupont[
        "fecha"
    ]
    .dt.month
)


fact_dupont[
    "año_mes"
] = (
    fact_dupont[
        "fecha"
    ]
    .dt.strftime("%Y-%m")
)


fact_dupont[
    "escenario"
] = "Real"


fact_dupont[
    "tipo_periodo"
] = "Histórico"


print(
    f"Movimientos contables: "
    f"{len(fact_contabilidad):,}"
)

print(
    f"Periodos DuPont: "
    f"{len(fact_dupont):,}"
)


display(
    fact_dupont.tail()
)

Movimientos contables: 78,688
Periodos DuPont: 36


,fecha,roe,dupont_margen_neto,dupont_rotacion_activos,dupont_apalancamiento,roe_año_anterior,dupont_margen_neto_año_anterior,dupont_rotacion_activos_año_anterior,dupont_apalancamiento_año_anterior,roe_variacion,efecto_margen,efecto_rotacion,efecto_apalancamiento,variacion_explicada,diferencia_descomposicion,principal_causa_variacion_roe,año,mes_numero,año_mes,escenario,tipo_periodo
31,2023-08-01,1.21,0.24,0.58,8.61,0.48,0.04,0.52,25.33,0.72,2.72,0.35,-2.34,0.72,0.00,Margen neto,2023,8,2023-08,Real,Histórico
32,2023-09-01,1.03,0.21,0.58,8.27,0.21,0.02,0.52,25.77,0.83,2.68,0.32,-2.18,0.83,0.00,Margen neto,2023,9,2023-09,Real,Histórico
33,2023-10-01,0.93,0.20,0.58,7.99,0.54,0.04,0.54,25.29,0.39,2.20,0.21,-2.02,0.39,0.00,Margen neto,2023,10,2023-10,Real,Histórico
34,2023-11-01,0.87,0.19,0.58,7.75,0.86,0.06,0.55,24.11,0.01,1.72,0.13,-1.84,0.01,0.00,Apalancamiento,2023,11,2023-11,Real,Histórico
35,2023-12-01,0.79,0.18,0.57,7.52,0.79,0.06,0.54,23.14,-0.00,1.50,0.13,-1.63,-0.00,0.00,Apalancamiento,2023,12,2023-12,Real,Histórico


In [10]:
# ==========================================================
# 10. DIAGNÓSTICO EJECUTIVO
# ==========================================================


def texto_variacion(
    nombre,
    actual,
    anterior,
    mejor_si_sube=True
):

    if (
        pd.isna(actual)
        or pd.isna(anterior)
    ):
        return None


    diferencia = (
        actual
        - anterior
    )


    if abs(diferencia) < 0.0001:

        return (
            f"{nombre} se mantiene estable "
            "frente al año anterior."
        )


    mejora = (
        diferencia > 0
        if mejor_si_sube
        else diferencia < 0
    )


    direccion = (
        "mejora"
        if mejora
        else "empeora"
    )


    return (
        f"{nombre} {direccion} frente al año anterior "
        f"({diferencia:+.2f})."
    )


# ==========================================================
# RATIOS UTILIZADOS
# ==========================================================

reglas = [
    (
        "roe",
        "YTD",
        "El ROE",
        True,
    ),

    (
        "margen_neto",
        "YTD",
        "El margen neto",
        True,
    ),

    (
        "liquidez_corriente",
        "CIERRE",
        "La liquidez corriente",
        True,
    ),

    (
        "endeudamiento",
        "CIERRE",
        "El endeudamiento",
        False,
    ),

    (
        "dias_cobro",
        "YTD",
        "El periodo medio de cobro",
        False,
    ),
]


diagnosticos = []


for fecha in sorted(
    ratios_historicos[
        "periodo"
    ].unique()
):

    datos_mes = (
        ratios_historicos.loc[
            ratios_historicos[
                "periodo"
            ].eq(fecha)
        ]
    )


    textos = []


    for (
        ratio,
        horizonte,
        nombre,
        mejor_si_sube
    ) in reglas:

        fila = datos_mes.loc[
            datos_mes[
                "ratio"
            ].eq(ratio)
            &
            datos_mes[
                "horizonte"
            ].eq(horizonte)
        ]


        if fila.empty:
            continue


        actual = (
            fila.iloc[0][
                "valor"
            ]
        )

        anterior = (
            fila.iloc[0][
                "valor_año_anterior"
            ]
        )


        texto = texto_variacion(
            nombre,
            actual,
            anterior,
            mejor_si_sube
        )


        if texto:
            textos.append(
                texto
            )


    diagnosticos.append({
        "fecha":
            fecha,

        "diagnostico":
            " ".join(
                textos[:4]
            ),

        "numero_alertas":
            len(
                datos_mes.loc[
                    datos_mes[
                        "estado"
                    ].eq(
                        "DESFAVORABLE"
                    )
                ]
            ),
    })


fact_diagnostico = (
    pd.DataFrame(
        diagnosticos
    )
)


display(
    fact_diagnostico.tail()
)

,fecha,diagnostico,numero_alertas
31,2023-08-01,El ROE mejora frente al año anterior (+0.72). ...,5
32,2023-09-01,El ROE mejora frente al año anterior (+0.83). ...,2
33,2023-10-01,El ROE mejora frente al año anterior (+0.39). ...,6
34,2023-11-01,El ROE mejora frente al año anterior (+0.01). ...,5
35,2023-12-01,El ROE empeora frente al año anterior (-0.00)....,7


In [11]:
# ==========================================================
# 11. PRINCIPALES ALERTAS
# ==========================================================


alertas_base = (
    ratios_historicos.loc[
        ratios_historicos[
            "estado"
        ].eq(
            "DESFAVORABLE"
        )
    ]
    .copy()
)


alertas_base[
    "impacto"
] = (
    alertas_base[
        "variacion_anual"
    ]
    .abs()
)


fact_alertas = (
    alertas_base
    .sort_values(
        [
            "periodo",
            "impacto",
        ],
        ascending=[
            True,
            False,
        ]
    )
    .groupby(
        "periodo",
        as_index=False,
        group_keys=False
    )
    .head(3)
    .rename(
        columns={
            "periodo":
                "fecha",

            "valor_año_anterior":
                "valor_ly",

            "variacion_anual":
                "variacion_abs",

            "variacion_anual_pct":
                "variacion_pct",
        }
    )
)


columnas_alertas = [
    "fecha",
    "ratio",
    "horizonte",
    "valor",
    "valor_ly",
    "variacion_abs",
    "variacion_pct",
    "estado",
    "grupo",
    "interpretacion",
]


columnas_alertas_disponibles = [
    columna
    for columna in columnas_alertas
    if columna
    in fact_alertas.columns
]


fact_alertas = (
    fact_alertas[
        columnas_alertas_disponibles
    ]
    .reset_index(drop=True)
)


display(
    fact_alertas.tail(10)
)

,fecha,ratio,horizonte,valor,valor_ly,variacion_abs,variacion_pct,estado,grupo,interpretacion
48,2023-09-01,margen_bruto,YTD,0.98,0.98,-0.00,-0.00,DESFAVORABLE,Rentabilidad,Margen después de aprovisionamientos
49,2023-10-01,margen_ebit,MTD,0.11,0.27,-0.16,-0.60,DESFAVORABLE,Rentabilidad,Rentabilidad operativa después de amortizaciones
50,2023-10-01,margen_ebitda,MTD,0.24,0.38,-0.14,-0.36,DESFAVORABLE,Rentabilidad,Rentabilidad operativa antes de amortizaciones
51,2023-10-01,margen_neto,MTD,0.08,0.21,-0.13,-0.63,DESFAVORABLE,Rentabilidad,Beneficio final generado por las ventas
52,2023-11-01,margen_ebit,MTD,0.15,0.35,-0.19,-0.55,DESFAVORABLE,Rentabilidad,Rentabilidad operativa después de amortizaciones
53,2023-11-01,margen_ebitda,MTD,0.28,0.45,-0.17,-0.38,DESFAVORABLE,Rentabilidad,Rentabilidad operativa antes de amortizaciones
54,2023-11-01,margen_neto,MTD,0.12,0.26,-0.14,-0.55,DESFAVORABLE,Rentabilidad,Beneficio final generado por las ventas
55,2023-12-01,margen_ebitda,MTD,0.37,0.50,-0.13,-0.26,DESFAVORABLE,Rentabilidad,Rentabilidad operativa antes de amortizaciones
56,2023-12-01,margen_ebit,MTD,0.22,0.34,-0.12,-0.36,DESFAVORABLE,Rentabilidad,Rentabilidad operativa después de amortizaciones
57,2023-12-01,margen_neto,MTD,0.02,0.04,-0.02,-0.50,DESFAVORABLE,Rentabilidad,Beneficio final generado por las ventas


In [12]:
# ==========================================================
# 12. VALIDACIÓN Y EXPORTACIÓN PARA POWER BI
# ==========================================================

# ==========================================================
# VALIDACIONES
# ==========================================================

if dim_calendario[
    "fecha"
].duplicated().any():

    raise ValueError(
        "Dim_Calendario contiene fechas duplicadas."
    )


if dim_cuentas[
    "cuenta_n5"
].duplicated().any():

    raise ValueError(
        "Dim_Cuentas contiene cuentas duplicadas."
    )


if dim_ratios[
    "ratio"
].duplicated().any():

    raise ValueError(
        "Dim_Ratios contiene ratios duplicados."
    )


if dim_escenarios[
    "escenario"
].duplicated().any():

    raise ValueError(
        "Dim_Escenarios contiene escenarios duplicados."
    )


if fact_finanzas[
    "valor"
].isin(
    [
        np.inf,
        -np.inf,
    ]
).any():

    raise ValueError(
        "Fact_Finanzas contiene infinitos."
    )


if fact_ratios[
    "valor"
].isin(
    [
        np.inf,
        -np.inf,
    ]
).any():

    raise ValueError(
        "Fact_Ratios contiene infinitos."
    )


# ==========================================================
# EXPORTACIONES
# ==========================================================

EXPORTACIONES = {

    "Dim_Calendario.csv":
        dim_calendario,

    "Dim_Cuentas.csv":
        dim_cuentas,

    "Dim_CentrosCoste.csv":
        dim_centros_coste,

    "Dim_Ratios.csv":
        dim_ratios,

    "Dim_Escenarios.csv":
        dim_escenarios,

    "Fact_Finanzas.csv":
        fact_finanzas,

    "Fact_Ratios.csv":
        fact_ratios,

    "Fact_Contabilidad.csv":
        fact_contabilidad,

    "Fact_CentrosCoste.csv":
        fact_centros_coste,

    "Fact_Dupont.csv":
        fact_dupont,

    "Fact_Diagnostico.csv":
        fact_diagnostico,

    "Fact_Alertas.csv":
        fact_alertas,

    "Fact_ResumenEscenarios.csv":
        resumen_escenarios,
}


for nombre, df in EXPORTACIONES.items():

    df.to_csv(
        POWERBI_DIR / nombre,
        index=False,
        encoding="utf-8-sig",
        decimal="."
    )


# ==========================================================
# RESUMEN
# ==========================================================

print(
    "Archivos exportados en data/powerbi:"
)

for nombre, df in EXPORTACIONES.items():

    print(
        f" - {nombre}: "
        f"{len(df):,} filas"
    )


print("\n" + "=" * 55)
print("NB6 COMPLETADO CORRECTAMENTE")
print("=" * 55)

print(
    f"Histórico hasta: "
    f"{ultima_fecha_real:%Y-%m}"
)

print(
    f"Forecast hasta: "
    f"{forecast['periodo'].max():%Y-%m}"
)

print(
    f"Escenarios forecast: "
    f"{forecast['escenario'].nunique()}"
)

print("=" * 55)

Archivos exportados en data/powerbi:
 - Dim_Calendario.csv: 48 filas
 - Dim_Cuentas.csv: 1,565 filas
 - Dim_CentrosCoste.csv: 30 filas
 - Dim_Ratios.csv: 24 filas
 - Dim_Escenarios.csv: 4 filas
 - Fact_Finanzas.csv: 1,476 filas
 - Fact_Ratios.csv: 1,512 filas
 - Fact_Contabilidad.csv: 78,688 filas
 - Fact_CentrosCoste.csv: 6,171 filas
 - Fact_Dupont.csv: 36 filas
 - Fact_Diagnostico.csv: 36 filas
 - Fact_Alertas.csv: 58 filas
 - Fact_ResumenEscenarios.csv: 3 filas

NB6 COMPLETADO CORRECTAMENTE
Histórico hasta: 2023-12
Forecast hasta: 2024-12
Escenarios forecast: 3
